# Lấy các cột cần thiết từ 3 file parquet

In [2]:
import pandas as pd
import os

In [3]:
train_path = "../data/raw/train_115-00000-of-00001.parquet"
val1_path = "../data/raw/validation-00000-of-00002.parquet"
val2_path = "../data/raw/validation-00001-of-00002.parquet"

In [4]:
train = pd.read_parquet(train_path)
val1 = pd.read_parquet(val1_path)
val2 = pd.read_parquet(val2_path)

print("Train:", train.shape)
print("Validation 1:", val1.shape)
print("Validation 2:", val2.shape)

Train: (115, 26)
Validation 1: (1017, 26)
Validation 2: (1016, 26)


In [5]:
print(train.columns.tolist())

['id', 'locale', 'partition', 'scenario', 'scenario_str', 'intent_idx', 'intent_str', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments', 'tokens', 'labels', 'audio', 'path', 'is_transcript_reported', 'is_validated', 'speaker_id', 'speaker_sex', 'speaker_age', 'speaker_ethnicity_simple', 'speaker_country_of_birth', 'speaker_country_of_residence', 'speaker_nationality', 'speaker_first_language']


In [6]:
train["original_split"] = "train"
val1["original_split"] = "validation"
val2["original_split"] = "validation"

In [7]:
dataset = pd.concat(
    [train, val1, val2],
    ignore_index=True
)

print(dataset.shape)
print(dataset["original_split"].value_counts())
print("Số speaker:", dataset["speaker_id"].nunique())

(2148, 27)
original_split
validation    2033
train          115
Name: count, dtype: int64
Số speaker: 35


In [11]:
inventory = dataset[[
    "id",
    "audio",
    "path",
    "original_split",
    "speaker_id",
    "utt",
    "annot_utt",
    "intent_idx",
    "intent_str",
    "scenario_str",
    "locale",
    "tokens",
    "labels",
    "speaker_sex",
    "speaker_age",
    "is_validated"
]].copy()

inventory.columns = [
    "audio_id",
    "audio",
    "audio_path",
    "original_split",
    "speaker_id",
    "transcript",
    "annot_utt",
    "intent_idx",
    "intent",
    "scenario_str",
    "locale",
    "tokens",
    "labels",
    "speaker_sex",
    "speaker_age",
    "is_valid"
]
print(inventory.columns.tolist())

['audio_id', 'audio', 'audio_path', 'original_split', 'speaker_id', 'transcript', 'annot_utt', 'intent_idx', 'intent', 'scenario_str', 'locale', 'tokens', 'labels', 'speaker_sex', 'speaker_age', 'is_valid']


In [12]:
inventory["project_split"] = None
inventory["normalized_speaker_id"] = None
inventory["protocol"] = None
inventory["role"] = None
inventory["checksum"] = None
print(inventory.columns.tolist())

['audio_id', 'audio', 'audio_path', 'original_split', 'speaker_id', 'transcript', 'annot_utt', 'intent_idx', 'intent', 'scenario_str', 'locale', 'tokens', 'labels', 'speaker_sex', 'speaker_age', 'is_valid', 'project_split', 'normalized_speaker_id', 'protocol', 'role', 'checksum']


In [13]:
import io
import soundfile as sf
import numpy as np

durations = []
sample_rates = []
num_channels = []

for audio in dataset["audio"]:
    try:
        waveform, sr = sf.read(io.BytesIO(audio["bytes"]))

        durations.append(len(waveform) / sr)
        sample_rates.append(sr)

        if waveform.ndim == 1:
            num_channels.append(1)
        else:
            num_channels.append(waveform.shape[1])

    except Exception:
        durations.append(np.nan)
        sample_rates.append(np.nan)
        num_channels.append(np.nan)

inventory["duration_sec"] = durations
inventory["sample_rate"] = sample_rates
inventory["num_channels"] = num_channels

print("\nDuration statistics:")
print(inventory["duration_sec"].describe())
print("\nSample rates:")
print(inventory["sample_rate"].value_counts())
print("\nChannels:")
print(inventory["num_channels"].value_counts())


Duration statistics:
count    2148.000000
mean        3.706076
std         1.777330
min         0.720000
25%         2.520000
50%         3.360000
75%         4.440000
max        15.955833
Name: duration_sec, dtype: float64

Sample rates:
sample_rate
48000    2148
Name: count, dtype: int64

Channels:
num_channels
1    2148
Name: count, dtype: int64


In [14]:
print("Missing duration:",
      inventory["duration_sec"].isna().sum())
print("Missing sample rate:",
      inventory["sample_rate"].isna().sum())
print("Missing channels:",
      inventory["num_channels"].isna().sum())

Missing duration: 0
Missing sample rate: 0
Missing channels: 0


In [17]:
inventory = inventory.drop(columns=["audio"])
print(inventory.columns.tolist())

['audio_id', 'audio_path', 'original_split', 'speaker_id', 'transcript', 'annot_utt', 'intent_idx', 'intent', 'scenario_str', 'locale', 'tokens', 'labels', 'speaker_sex', 'speaker_age', 'is_valid', 'project_split', 'normalized_speaker_id', 'protocol', 'role', 'checksum', 'duration_sec', 'sample_rate', 'num_channels']


In [18]:
print(inventory.head())
print(inventory.shape)
print(inventory.isnull().sum())

  audio_id                                      audio_path original_split  \
0     9702  train-115/4dcf89cc7708ffe6339d97afd4da24f5.wav          train   
1     9671  train-115/2d48e259c29bdbf2039edfddad79cb61.wav          train   
2    10249  train-115/81da41ae4fbf09485ad6a4214439f9d0.wav          train   
3     3854  train-115/2866444482800feff8590246b56cd245.wav          train   
4    12053  train-115/ddd9fc1eee9336973c9de546c7af4794.wav          train   

                 speaker_id  \
0  657c8d982832af573ef2c039   
1  5cf03d69b094d700013e4d54   
2  5e25be7c5514e680ef436338   
3  657c8d982832af573ef2c039   
4  659ea35db097ea3c414b04c0   

                                          transcript  \
0     tôi muốn nghe một quyển sách bởi la quán trung   
1  bắt đầu phát tam quốc diễn nghĩa ở chỗ mà tôi ...   
2                         hãy chơi một ván trivia   
3                                   tắt loa làm ơn   
4  các bộ phim được đánh giá cao đang chiếu cuối ...   

             

In [19]:
# Reorder columns
inventory = inventory[
    [
        # Audio identity
        "audio_id",
        "audio_path",

        # Dataset information
        "original_split",
        "project_split",
        "locale",

        # Speaker information
        "speaker_id",
        "normalized_speaker_id",
        "speaker_sex",
        "speaker_age",

        # Transcript & annotation
        "transcript",
        "annot_utt",
        "tokens",
        "labels",

        # Intent information
        "intent_idx",
        "intent",
        "scenario_str",

        # Audio metadata
        "duration_sec",
        "sample_rate",
        "num_channels",

        # Validation
        "is_valid",

        # Project metadata
        "protocol",
        "role",
        "checksum"
    ]
]

print("Columns reordered successfully.")
print(inventory.columns.tolist())

Columns reordered successfully.
['audio_id', 'audio_path', 'original_split', 'project_split', 'locale', 'speaker_id', 'normalized_speaker_id', 'speaker_sex', 'speaker_age', 'transcript', 'annot_utt', 'tokens', 'labels', 'intent_idx', 'intent', 'scenario_str', 'duration_sec', 'sample_rate', 'num_channels', 'is_valid', 'protocol', 'role', 'checksum']


# Chia project split

In [38]:
speaker_stats = (
    inventory
    .groupby("speaker_id")
    .agg(
        num_audio=("audio_id", "count"),
        total_duration=("duration_sec", "sum"),
        speaker_sex=("speaker_sex", "first"),
        speaker_age=("speaker_age", "first")
    )
    .reset_index()
)

speaker_stats = speaker_stats.sort_values(
    "num_audio",
    ascending=False
)

display(speaker_stats.head(10))

speaker_stats.to_csv(
    "../data/metadata/speaker_distribution.csv",
    index=False,
    encoding="utf-8-sig"
)

,speaker_id,num_audio,total_duration,speaker_sex,speaker_age
28,6552b16899091e4aaa587d1b,151,494.820000,Male,33
23,6537e43c008bb605430a302f,144,341.160000,Female,29
1,5c9d67af9e01ef000197cd15,100,387.600000,Female,59
3,5e133bbe6a5bff9911b73cfc,100,343.680000,Female,31
29,6557ebf8541bd32382739c1a,100,464.940000,Female,30
6,5ee49e71f8d19d00098aca3e,99,334.140000,Male,27
5,5e4247e718b8910d4a2b9a34,99,408.660000,Female,32
9,5fb84feecf0626000b30cbd3,99,291.660000,Male,29
10,604b93f6432ad99bcb4798e8,98,659.455667,Male,31
18,63cfbea470ab1bba9db30082,97,238.800000,Male,21


In [39]:
eligible = speaker_stats[
    speaker_stats["num_audio"] >= 25
]

print("Eligible speakers:", len(eligible))

display(eligible)

Eligible speakers: 27


,speaker_id,num_audio,total_duration,speaker_sex,speaker_age
28,6552b16899091e4aaa587d1b,151,494.820000,Male,33
23,6537e43c008bb605430a302f,144,341.160000,Female,29
1,5c9d67af9e01ef000197cd15,100,387.600000,Female,59
3,5e133bbe6a5bff9911b73cfc,100,343.680000,Female,31
29,6557ebf8541bd32382739c1a,100,464.940000,Female,30
6,5ee49e71f8d19d00098aca3e,99,334.140000,Male,27
5,5e4247e718b8910d4a2b9a34,99,408.660000,Female,32
9,5fb84feecf0626000b30cbd3,99,291.660000,Male,29
10,604b93f6432ad99bcb4798e8,98,659.455667,Male,31
18,63cfbea470ab1bba9db30082,97,238.800000,Male,21


In [40]:
exp_svm = eligible.iloc[0:10].copy()
val_enrolled = eligible.iloc[10:12].copy()
val_unknown = eligible.iloc[12:14].copy()
test_enrolled = eligible.iloc[14:16].copy()
test_unknown = eligible.iloc[16:18].copy()

In [41]:
# Experimental SVM
exp_svm["normalized_speaker_id"] = [
    f"exp_svm_spk_{i:04d}"
    for i in range(1, 11)
]

exp_svm["protocol"] = "SVM_CLOSED_SET"
exp_svm["role"] = "SVM_EXPERIMENTAL"
exp_svm["project_split"] = "SVM"


# Validation enrolled
val_enrolled["normalized_speaker_id"] = [
    "val_enrolled_spk_0001",
    "val_enrolled_spk_0002"
]

val_enrolled["protocol"] = "COSINE_VALIDATION"
val_enrolled["role"] = "ENROLLED"
val_enrolled["project_split"] = "VALIDATION"


# Validation unknown
val_unknown["normalized_speaker_id"] = [
    "val_unknown_spk_0001",
    "val_unknown_spk_0002"
]

val_unknown["protocol"] = "COSINE_VALIDATION"
val_unknown["role"] = "UNKNOWN"
val_unknown["project_split"] = "VALIDATION"


# Test enrolled
test_enrolled["normalized_speaker_id"] = [
    "test_enrolled_spk_0001",
    "test_enrolled_spk_0002"
]

test_enrolled["protocol"] = "COSINE_TEST"
test_enrolled["role"] = "ENROLLED"
test_enrolled["project_split"] = "TEST"


# Test unknown
test_unknown["normalized_speaker_id"] = [
    "test_unknown_spk_0001",
    "test_unknown_spk_0002"
]

test_unknown["protocol"] = "COSINE_TEST"
test_unknown["role"] = "UNKNOWN"
test_unknown["project_split"] = "TEST"

In [42]:
mapping = pd.concat(
    [
        exp_svm,
        val_enrolled,
        val_unknown,
        test_enrolled,
        test_unknown
    ],
    ignore_index=True
)

mapping = mapping[
    [
        "speaker_id",
        "normalized_speaker_id",
        "protocol",
        "role",
        "project_split",
        "num_audio"
    ]
]

display(mapping)

,speaker_id,normalized_speaker_id,protocol,role,project_split,num_audio
0,6552b16899091e4aaa587d1b,exp_svm_spk_0001,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,151
1,6537e43c008bb605430a302f,exp_svm_spk_0002,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,144
2,5c9d67af9e01ef000197cd15,exp_svm_spk_0003,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,100
3,5e133bbe6a5bff9911b73cfc,exp_svm_spk_0004,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,100
4,6557ebf8541bd32382739c1a,exp_svm_spk_0005,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,100
5,5ee49e71f8d19d00098aca3e,exp_svm_spk_0006,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,99
6,5e4247e718b8910d4a2b9a34,exp_svm_spk_0007,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,99
7,5fb84feecf0626000b30cbd3,exp_svm_spk_0008,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,99
8,604b93f6432ad99bcb4798e8,exp_svm_spk_0009,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,98
9,63cfbea470ab1bba9db30082,exp_svm_spk_0010,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM,97


In [43]:
inventory["normalized_speaker_id"] = pd.NA
inventory["protocol"] = pd.NA
inventory["role"] = pd.NA

# Các speaker chưa dùng
inventory["project_split"] = "UNUSED"

for _, row in mapping.iterrows():

    mask = inventory["speaker_id"] == row["speaker_id"]

    inventory.loc[mask, "normalized_speaker_id"] = row["normalized_speaker_id"]

    inventory.loc[mask, "protocol"] = row["protocol"]

    inventory.loc[mask, "role"] = row["role"]

    inventory.loc[mask, "project_split"] = row["project_split"]

In [44]:
print(inventory["project_split"].value_counts())

display(
    inventory[
        [
            "speaker_id",
            "normalized_speaker_id",
            "protocol",
            "role",
            "project_split"
        ]
    ].drop_duplicates()
)

project_split
SVM           1087
UNUSED         472
VALIDATION     335
TEST           254
Name: count, dtype: int64


,speaker_id,normalized_speaker_id,protocol,role,project_split
0,657c8d982832af573ef2c039,<NA>,<NA>,<NA>,UNUSED
1,5cf03d69b094d700013e4d54,<NA>,<NA>,<NA>,UNUSED
2,5e25be7c5514e680ef436338,<NA>,<NA>,<NA>,UNUSED
4,659ea35db097ea3c414b04c0,<NA>,<NA>,<NA>,UNUSED
5,65a242d84b379f4cebea1b21,<NA>,<NA>,<NA>,UNUSED
7,65718bf0ad0f780a67a20790,<NA>,<NA>,<NA>,UNUSED
64,5f888877136ad50208b48b47,<NA>,<NA>,<NA>,UNUSED
115,5fb84feecf0626000b30cbd3,exp_svm_spk_0008,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM
116,60d3432bc679834595ca8139,<NA>,<NA>,<NA>,UNUSED
117,5e133bbe6a5bff9911b73cfc,exp_svm_spk_0004,SVM_CLOSED_SET,SVM_EXPERIMENTAL,SVM


In [45]:
mapping.to_csv(
    "../data/metadata/speaker_id_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

In [46]:
print(speaker_stats.columns.tolist())

['speaker_id', 'num_audio', 'total_duration', 'speaker_sex', 'speaker_age']


In [47]:
output_dir = "../data/metadata"
os.makedirs(output_dir, exist_ok=True)

# Chỉ giữ các cột cần thiết
columns = [
    "speaker_id",
    "normalized_speaker_id",
    "protocol",
    "role",
    "project_split",
    "num_audio",
    "speaker_sex",
    "speaker_age"
]

# Experimental SVM
exp_svm[columns].to_csv(
    f"{output_dir}/selected_svm_experimental_speakers.csv",
    index=False,
    encoding="utf-8-sig"
)

# Validation enrolled
val_enrolled[columns].to_csv(
    f"{output_dir}/selected_validation_enrolled_speakers.csv",
    index=False,
    encoding="utf-8-sig"
)

# Validation unknown
val_unknown[columns].to_csv(
    f"{output_dir}/selected_validation_unknown_speakers.csv",
    index=False,
    encoding="utf-8-sig"
)

# Test enrolled
test_enrolled[columns].to_csv(
    f"{output_dir}/selected_test_enrolled_speakers.csv",
    index=False,
    encoding="utf-8-sig"
)

# Test unknown
test_unknown[columns].to_csv(
    f"{output_dir}/selected_test_unknown_speakers.csv",
    index=False,
    encoding="utf-8-sig"
)

print("All selected speaker files have been saved.")

All selected speaker files have been saved.


# Lưu file

In [34]:
os.makedirs("../data/metadata", exist_ok=True)

output_path = "../data/metadata/data_inventory.csv"

inventory.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)

Saved: ../data/metadata/data_inventory.csv


In [35]:
check = pd.read_csv(output_path)

print(check.shape)
print(check.columns.tolist())
print(check.head())

(2148, 23)
['audio_id', 'audio_path', 'original_split', 'project_split', 'locale', 'speaker_id', 'normalized_speaker_id', 'speaker_sex', 'speaker_age', 'transcript', 'annot_utt', 'tokens', 'labels', 'intent_idx', 'intent', 'scenario_str', 'duration_sec', 'sample_rate', 'num_channels', 'is_valid', 'protocol', 'role', 'checksum']
   audio_id                                      audio_path original_split  \
0      9702  train-115/4dcf89cc7708ffe6339d97afd4da24f5.wav          train   
1      9671  train-115/2d48e259c29bdbf2039edfddad79cb61.wav          train   
2     10249  train-115/81da41ae4fbf09485ad6a4214439f9d0.wav          train   
3      3854  train-115/2866444482800feff8590246b56cd245.wav          train   
4     12053  train-115/ddd9fc1eee9336973c9de546c7af4794.wav          train   

  project_split locale                speaker_id normalized_speaker_id  \
0        UNUSED  vi-VN  657c8d982832af573ef2c039                   NaN   
1        UNUSED  vi-VN  5cf03d69b094d700013e4d54     